# ENV Simulation — Colab Runner

This notebook clones the simulation code from GitHub, installs dependencies,
mounts Google Drive for persistence, starts the simulation + FastAPI server,
and exposes the dashboard via Cloudflare Tunnel.

**Repository**: https://github.com/kkdkavindu-beep/ENV-Simulation

**Output**: Logs and checkpoints saved to `/content/drive/MyDrive/envsim/`

In [ ]:
# ── Cell 1: Install Dependencies ────────────────────────────────────────────
!pip install -q fastapi uvicorn[standard] orjson numba

# Optional GPU (uncomment if T4 available)
# !pip install -q cupy-cuda12x

print("✅ Dependencies installed")

In [ ]:
# ── Cell 2: Deterministic RNG ──────────────────────────────────────────────
import numpy as np
import random

SEED = 42  # Change for different runs
rng = np.random.default_rng(SEED)
random.seed(SEED)

# Numba cache directory (persists across restarts)
import numba
numba.config.CACHE_DIR = "/content/drive/MyDrive/envsim/.numba_cache"

print(f"✅ RNG seeded: {SEED}")

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = "/content/drive/MyDrive/envsim"
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, ".numba_cache"), exist_ok=True)

# Verify write access
_test = os.path.join(DRIVE_BASE, ".write_test")
try:
    with open(_test, 'w') as f:
        f.write("ok")
    os.remove(_test)
    print("✅ Google Drive mounted and writable")
except Exception as e:
    print(f"⚠️ Drive write failed: {e}")
    DRIVE_BASE = "/content/envsim"
    os.makedirs(DRIVE_BASE, exist_ok=True)

In [ ]:
# ── Cell 4: Clone Repository & Setup Python Path ───────────────────────────
import subprocess, sys, os

REPO_URL = "https://github.com/kkdkavindu-beep/ENV-Simulation.git"
REPO_DIR = "/content/ENV-Simulation"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Repository cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✅ Repository updated")

# Try multiple possible locations for the 'systems' package
possible_paths = [
    os.path.join(REPO_DIR, "code", "code"),   # nested code/code (current structure)
    os.path.join(REPO_DIR, "code"),             # flat code/ (alt structure)
    "/content/ENV-Simulation/code/code",        # absolute fallback
    "/content/ENV-Simulation/code",             # absolute fallback
]

CODE_DIR = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, "systems", "__init__.py")):
        CODE_DIR = p
        break

if CODE_DIR is None:
    raise RuntimeError(
        f"Could not find 'systems' package. Tried:
" +
        "
".join(f"  {p}" for p in possible_paths) +
        f"
Contents of {REPO_DIR}: {os.listdir(REPO_DIR) if os.path.exists(REPO_DIR) else 'NOT FOUND'}"
    )

sys.path.insert(0, CODE_DIR)
print(f"✅ Python path: {CODE_DIR}")
print(f"   systems importable: {os.path.exists(os.path.join(CODE_DIR, 'systems', '__init__.py'))}")


In [ ]:
# ── Cell 5: Cloudflare Tunnel Setup ────────────────────────────────────────
import subprocess, threading, re, time

CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/"
    "download/cloudflared-linux-amd64"
)

def install_cloudflared():
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", CLOUDFLARED_URL], check=True)
        subprocess.run(["chmod", "+", "/usr/local/bin/cloudflared"], check=True)
        print("✅ cloudflared downloaded")
    else:
        print("✅ cloudflared already installed")

install_cloudflared()

tunnel_url = None
tunnel_lock = threading.Lock()

def start_tunnel(port: int = 8080) -> None:
    global tunnel_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}",
         "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            with tunnel_lock:
                tunnel_url = match.group(0)
            print(f"\n🌐 Public URL: {tunnel_url}\n")
            break

print("Tunnel will start after server...")

In [ ]:
# ── Cell 6: Initialize Simulation ──────────────────────────────────────────
import numpy as np
import threading, time, json, pickle, zlib
from datetime import datetime
from collections import defaultdict
from numba import jit

# Import all systems
from systems import (
    config, rng, animal_model, genome_traits, mutation_repro,
    brain_nn, detection, sound, energy, fatigue, memory,
    world, tick_loop, logging_module, server
)

# Initialize RNG in all modules
from systems.rng import init_rng
init_rng(SEED)

# Initialize world
from systems.world import init_world
init_world(seed=SEED)

# Spawn progenitors
from systems.animal_model import reset_arrays
reset_arrays()
init_world(seed=SEED)

from systems.tick_loop import _spawn_progenitors
_spawn_progenitors(n_herb=20, n_carn=5)

# Initialize logging
from systems.logging_module import LogWriter, write_schema_headers
RUN_ID = datetime.now().strftime("%Y-%m-%d_%H%M%S")
RUN_DIR = os.path.join(DRIVE_BASE, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)

logging_module.ids_writer = LogWriter(os.path.join(RUN_DIR, "ids.jsonl"), buffer_lines=100)
logging_module.dynamic_writer = LogWriter(os.path.join(RUN_DIR, "dynamic.jsonl"), buffer_lines=2000)
write_schema_headers()

print(f"✅ Simulation initialized")
print(f"   Run ID: {RUN_ID}")
print(f"   Logs: {RUN_DIR}")

In [ ]:
# ── Cell 8: Start FastAPI Server ───────────────────────────────────────────
import time

def run_server():
    from systems.server import run_server as _run_server
    _run_server()

server_thread = threading.Thread(target=run_server, daemon=True, name="fastapi")
server_thread.start()
time.sleep(3)  # Wait for server to bind
print("✅ FastAPI server started on port 8080")


In [ ]:
# ── Cell 9: Start Cloudflare Tunnel ────────────────────────────────────────
tunnel_thread = threading.Thread(
    target=start_tunnel, args=(config.API_PORT,), daemon=True, name="cf-tunnel"
)
tunnel_thread.start()

# Wait for URL
for _ in range(30):
    with tunnel_lock:
        if tunnel_url:
            break
    time.sleep(0.5)

with tunnel_lock:
    if tunnel_url:
        print(f"🌍 Dashboard: {tunnel_url}")
    else:
        print("⚠️ Tunnel URL not found. Check cloudflared logs.")

In [ ]:
# ── Cell 10 (Alt): Gradio Dashboard (no tunnel needed) ──────────────
# Run this INSTEAD of cells 5, 8, 9 if Cloudflare fails
!pip install -q gradio
import gradio as gr
import threading, time, json

def get_status():
    snap = tick_loop.latest_snapshot
    if not snap:
        return "Waiting for simulation..."
    w = snap["world"]
    return f"Turn: {snap["turn"]} | Herb: {w["herbivore_count"]} | Carn: {w["carnivore_count"]} | Plants: {w["herb_count"]} | Carcasses: {w["carcass_count"]}"

def get_animals(species_filter="all"):
    snap = tick_loop.latest_snapshot
    if not snap:
        return []
    animals = []
    for a in snap["animals"]:
        if species_filter != "all" and a["species"] != species_filter:
            continue
        animals.append([a["id"], a["species"], f"{a["x"]:.1f}", f"{a["y"]:.1f}", f"{a["energy"]:.1f}", f"{a["age"]}", f"{a["generation"]}"])
    return animals

with gr.Blocks(title="ENV Simulation Dashboard") as demo:
    gr.Markdown("# 🌍 ENV Simulation Dashboard")
    status = gr.Textbox(label="Status", value="Loading...")
    species_dd = gr.Dropdown(["all", "herbivore", "carnivore"], value="all", label="Filter")
    animal_table = gr.Dataframe(headers=["ID", "Species", "X", "Y", "Energy", "Age", "Gen"], label="Animals")
    
    def refresh(species):
        return get_status(), get_animals(species)
    
    demo.load(refresh, inputs=[species_dd], outputs=[status, animal_table])
    species_dd.change(refresh, inputs=[species_dd], outputs=[status, animal_table])
    
    # Auto-refresh every 3s
    timer = gr.Timer(3)
    timer.tick(refresh, inputs=[species_dd], outputs=[status, animal_table])

# Launch on port 7860 (Gradio default), accessible via Colab proxy
demo.launch(server_name="0.0.0.0", server_port=7860, share=False, quiet=True)


In [ ]:
# ── Cell 10: Live Monitor (Optional) ───────────────────────────────────────
from IPython.display import clear_output

def monitor(duration_sec: int = 60, interval: float = 2.0):
    """Print live stats for specified duration."""
    end_time = time.time() + duration_sec
    while time.time() < end_time and not tick_loop.sim_stop.is_set():
        clear_output(wait=True)
        snap = tick_loop.latest_snapshot
        if snap:
            print(f"Turn: {snap['turn']:>8}  Herbivores: {snap['world']['herbivore_count']:>4}  Carnivores: {snap['world']['carnivore_count']:>4}")
            print(f"Plants: {snap['world']['herb_count']:>4}  Carcasses: {snap['world']['carcass_count']:>3}  Obstacles: {snap['world']['obstacle_count']:>3}")
            print(f"Speed: {snap.get('sim_speed', 0)} t/s  Paused: {snap.get('paused', False)}")
        else:
            print("Waiting for first snapshot...")
        time.sleep(interval)

# Run for 2 minutes
monitor(120, 2.0)

In [ ]:
# ── Cell 11: Manual Controls (run as needed) ───────────────────────────────
# Pause simulation
# tick_loop.sim_paused.set()

# Resume simulation
# tick_loop.sim_paused.clear()

# Change speed
# tick_loop.sim_speed = 100

# Stop simulation
# tick_loop.sim_stop.set()

# Force checkpoint
# tick_loop.checkpoint_to_drive(tick_loop.turn)

# Reseed RNG
# from systems.rng import reseed
# reseed(12345)

In [ ]:
# ── Cell 12: Resume from Checkpoint (if needed) ────────────────────────────
# def resume_from_checkpoint(run_id: str, turn_num: int):
#     """Load checkpoint and continue simulation."""
#     import pickle, zlib
#     
#     chk_path = os.path.join(DRIVE_BASE, run_id, "checkpoints",
#                            f"turn_{turn_num:08d}.pkl.zst")
#     with open(chk_path, 'rb') as f:
#         compressed = f.read()
#     data = zlib.decompress(compressed)
#     state = pickle.loads(data)
#     
#     # Stop current sim
#     tick_loop.sim_stop.set()
#     time.sleep(0.5)
#     
#     # Restore arrays (simplified - full restore in tick_loop.py)
#     # This would require calling tick_loop._restore_checkpoint_state(state)
#     print(f"Resume from {run_id} turn {turn_num} - implement full restore")
# 
# # Example:
# # resume_from_checkpoint("2026-01-15_143022", 5000)